[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/21_dimensionality_reduction/90_exercises.ipynb)

# Bài tập — Chương 21: Dimensionality Reduction (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/21_dimensionality_reduction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)
digits = load_digits()

## L0 — Nhớ khái niệm

**Câu 1.** Viết bài toán tối ưu của PCA (thành phần thứ nhất), liên hệ với trị riêng/vector riêng; công thức explained variance ratio; liên hệ SVD ↔ PCA. t-SNE tối thiểu đại lượng gì?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một nhà phân tích trình bày biểu đồ t-SNE: "Cụm khách hàng VIP nhỏ và nằm xa hẳn các cụm khác, nên VIP rất khác biệt và đồng nhất". Nhận xét nào đúng, nào không có căn cứ?

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Ma trận hiệp phương sai $C = \begin{bmatrix}4 & 2\\2 & 1\end{bmatrix}$. Tìm trị riêng, vector riêng PC1 (chuẩn hoá), tỉ lệ phương sai PC1 giải thích.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
lam1, lam2 = None, None   # trị riêng (giải det(C − λI) = 0), lam1 ≥ lam2
pc1 = None                # vector riêng chuẩn hoá của lam1
ratio1 = None
### END CODE HERE ###

In [ ]:
C = np.array([[4, 2], [2, 1.0]])
vals, vecs = np.linalg.eigh(C)
check_close(sorted([lam1, lam2]), vals, tol=1e-9, message="trị riêng")
check_close(np.abs(pc1 @ vecs[:, -1]), 1.0, message="PC1 cùng phương với vector riêng lớn nhất")
check_close(ratio1, 1.0, message="PC1 giải thích 100% (dữ liệu nằm trên một đường thẳng)")

**Câu 4.** Dữ liệu 3D có trị riêng $[6, 3, 1]$. (a) Tỉ lệ phương sai giải thích của 2 thành phần đầu. (b) Sai số tái tạo trung bình (theo phương sai) khi chỉ giữ 1 thành phần. (c) Cần ít nhất bao nhiêu thành phần để giữ ≥ 85% phương sai?

In [ ]:
lam = np.array([6, 3, 1.0])
### START CODE HERE ### (≈ 3 dòng)
ratio_2 = None
recon_err_1 = None
k_85 = None
### END CODE HERE ###

In [ ]:
check_close(ratio_2, 0.9, message="2 thành phần giải thích 90%")
check_close(recon_err_1, 4.0, message="sai số = tổng trị riêng bị bỏ")
check_equal(k_85, 2, "cần 2 thành phần cho ≥ 85%")

## L3 — Code

**Bài 5.** Cài đặt `class MyPCA` với `fit` (qua SVD), `transform`, `inverse_transform`, thuộc tính `components_`, `explained_variance_ratio_`. Kiểm tra với sklearn trên digits (so sánh có tính đến dấu ±).

In [ ]:
class MyPCA:
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        ### START CODE HERE ### (≈ 6 dòng)
        # self.mean_; SVD của X − mean; var = S² / (n − 1)
        # self.components_ = n_components hàng đầu của Vᵀ; self.explained_variance_ratio_ = (var / var.sum())[:n_components]
        pass
        ### END CODE HERE ###
        return self

    def transform(self, X):
        ### START CODE HERE ### (≈ 1 dòng)
        return None   # (X − mean) @ componentsᵀ
        ### END CODE HERE ###

    def inverse_transform(self, Z):
        ### START CODE HERE ### (≈ 1 dòng)
        return None   # Z @ components + mean
        ### END CODE HERE ###

In [ ]:
Xd = digits.data
mine = MyPCA(10).fit(Xd)
sk = PCA(10).fit(Xd)
check_close(mine.explained_variance_ratio_, sk.explained_variance_ratio_, tol=1e-8, message="explained variance ratio")
check_close(np.abs((mine.components_ * sk.components_).sum(1)), np.ones(10), tol=1e-6, message="thành phần trùng (đến dấu)")
check_close(mine.inverse_transform(mine.transform(Xd[:5])), sk.inverse_transform(sk.transform(Xd[:5])), tol=1e-6, message="tái tạo khớp sklearn")

**Bài 6.** Viết `n_components_for_variance(X, target)` trả về số thành phần nhỏ nhất đạt phương sai tích luỹ ≥ `target`, và `reconstruction_mse(X, k)` (MSE tái tạo khi giữ $k$ thành phần). Vẽ MSE theo $k$ cho digits.

In [ ]:
def n_components_for_variance(X, target=0.9):
    ### START CODE HERE ### (≈ 2 dòng)
    ratio = None
    return None   # số thành phần nhỏ nhất có cumsum(ratio) ≥ target
    ### END CODE HERE ###


def reconstruction_mse(X, k):
    ### START CODE HERE ### (≈ 2 dòng)
    p = None
    return None   # MSE giữa X và inverse_transform(transform(X))
    ### END CODE HERE ###

In [ ]:
ks = [1, 2, 5, 10, 20, 40]
mses = [reconstruction_mse(Xd, k) for k in ks]
plt.plot(ks, mses, "o-")
plt.xlabel("k")
plt.ylabel("MSE tái tạo")
plt.show()
check_equal(n_components_for_variance(Xd, 0.95), PCA(0.95).fit(Xd).n_components_, "khớp PCA(0.95)")
check_true(all(np.diff(mses) < 0), "MSE giảm khi k tăng")

## L4 — Debug

**Bài 7.** Pipeline sau dùng PCA trước KNN cho dữ liệu khách hàng nhưng (1) PCA fit trên **toàn bộ** dữ liệu trước CV, (2) không chuẩn hoá (thu nhập theo VND áp đảo), (3) số thành phần cố định = 1. Sửa: Pipeline(StandardScaler → PCA → KNN), chọn `n_components` ∈ {1, 2, 3, 4} bằng GridSearchCV (5-fold). Lưu `best_nc` và `cv_acc`.

In [ ]:
seg = pd.read_csv(repo_path("data", "sample", "customer_segments.csv"))
X7 = seg[["annual_income_musd", "spending_score", "age", "visits_per_month"]].assign(annual_income_musd=lambda d: d.annual_income_musd * 1e6)
y7 = seg["true_segment"]
### START CODE HERE ### (≈ 6 dòng)
Z7 = PCA(1).fit_transform(X7)                                                  # 🐛 (1), (2), (3)
cv_acc = cross_val_score(KNeighborsClassifier(5), Z7, y7, cv=5).mean()
best_nc = 1
### END CODE HERE ###
print(f"n_components = {best_nc} | accuracy CV = {cv_acc:.3f}")

In [ ]:
check_greater(cv_acc, 0.9, "accuracy CV > 0.9")
check_greater(best_nc, 1, "cần hơn 1 thành phần")

## L5 — Phân tích

**Bài 8.** Khảo sát **trustworthiness** (k = 10) của PCA và t-SNE (perplexity ∈ {5, 30, 50}) trên 800 ảnh digits đầu; lưu vào dict `trust`. Thêm thí nghiệm: chạy t-SNE (perplexity 30) với 3 seed khác nhau và đo **khoảng cách giữa tâm chữ số 0 và chữ số 6** trong 2D (`dist_06`, list). Nhận xét về độ ổn định của khoảng cách giữa cụm.

In [ ]:
X8, y8 = digits.data[:800], digits.target[:800]
### START CODE HERE ### (≈ 10 dòng)
trust = None     # dict: "PCA" và "t-SNE p=5/30/50" → trustworthiness(X8, Z, n_neighbors=10)
dist_06 = None   # list 3 khoảng cách giữa tâm chữ số 0 và 6 (t-SNE perplexity 30, seed 0/1/2, bố cục đã chuẩn hoá)
### END CODE HERE ###
print({k: round(v, 3) for k, v in trust.items()}, "| khoảng cách tâm 0–6 qua các seed:", np.round(dist_06, 2))

In [ ]:
check_greater(min(v for k, v in trust.items() if k.startswith("t-SNE")), trust["PCA"], "t-SNE giữ láng giềng tốt hơn PCA")
check_equal(len(dist_06), 3, "3 seed")

✍️ Nhận xét:

## L6 — Thực tế

**Bài 9.** Nén ảnh khuôn mặt giả lập: tạo 400 "ảnh" 20×20 từ 6 "khuôn mẫu" ngẫu nhiên (mỗi ảnh = tổ hợp tuyến tính ngẫu nhiên của khuôn mẫu + nhiễu nhỏ). Dùng PCA để: (a) tìm số thành phần giữ ≥ 99% phương sai (`k99`); (b) tính tỉ lệ nén (số con số lưu trữ khi giữ `k99` thành phần, gồm cả các thành phần và mean, so với dữ liệu gốc) — `compression`; (c) MSE tái tạo tương đối (`rel_mse` = MSE / phương sai dữ liệu).

In [ ]:
templates = rng.normal(size=(6, 400))
coef = rng.normal(size=(400, 6))
faces = coef @ templates + rng.normal(0, 0.1, (400, 400))
### START CODE HERE ### (≈ 6 dòng)
p9 = None
k99 = None
stored = None
compression = None
rel_mse = None
### END CODE HERE ###
print(f"k99 = {k99} | lưu {compression:.1%} dữ liệu gốc | MSE tương đối = {rel_mse:.4f}")

In [ ]:
check_less(k99, 10, "cấu trúc 6 khuôn mẫu → cần rất ít thành phần")
check_less(compression, 0.1, "nén còn < 10%")
check_less(rel_mse, 0.01, "sai số tái tạo < 1% phương sai")

## L7 — Nâng cao

**Bài 10.** **Power iteration**: tìm PC1 không cần SVD đầy đủ — lặp $\mathbf w \leftarrow C\mathbf w / \lVert C\mathbf w\rVert$ (hội tụ về vector riêng trội). Sau đó **deflation**: $C \leftarrow C - \lambda_1\mathbf w_1\mathbf w_1^\top$ để tìm PC2. Cài đặt `top_k_components(X, k, n_iter)` và so với sklearn trên digits (k = 3).

In [ ]:
def top_k_components(X, k=3, n_iter=200, seed=0):
    r = np.random.default_rng(seed)
    Xc = X - X.mean(0)
    C = Xc.T @ Xc / (len(X) - 1)
    comps, vals = [], []
    ### START CODE HERE ### (≈ 8 dòng)
    # k lần: w ngẫu nhiên; lặp n_iter: w = C @ w rồi chuẩn hoá; lam = wᵀ C w; lưu w, lam
    # deflation: C = C − lam · w wᵀ
    pass
    ### END CODE HERE ###
    return np.array(comps), np.array(vals)

In [ ]:
comps10, vals10 = top_k_components(Xd, 3)
sk10 = PCA(3).fit(Xd)
check_close(vals10, sk10.explained_variance_, tol=1e-4, message="trị riêng khớp sklearn")
check_close(np.abs((comps10 * sk10.components_).sum(1)), np.ones(3), tol=1e-4, message="vector riêng khớp (đến dấu)")